# EDA — Prédiction du risque d'annulation
**Maeva — Mastère DIA Paris**

Analyse exploratoire du dataset `v_dataset_annulation`.
Objectif : comprendre la structure des données, la cible, et valider les signaux clés (condition Flex, canal, engagement email) avant la modélisation.

---

## 0. Installation (une seule fois)

In [ ]:
# %pip install pandas numpy matplotlib seaborn google-cloud-bigquery db-dtypes pyarrow

## 1. Imports & configuration

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 160)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 12

# Palette cohérente
C_ANNUL = '#e74c3c'   # annulation
C_MAINT = '#2ecc71'   # maintenu
C_BLUE  = '#3498db'
C_PURP  = '#9b59b6'

# Dossier pour sauvegarder les graphiques (pour le mémoire)
os.makedirs('figures', exist_ok=True)
print('Imports OK')

## 2. Chargement des données
Lit le CSV local en priorité ; si absent, recharge depuis BigQuery.

In [ ]:
CSV_PATH = 'data/dataset_annulation.csv'
VIEW     = 'groupe-lfdnas.data_hady.v_dataset_annulation'
PROJECT  = 'groupe-lfdnas'

if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH, encoding='utf-8-sig')
    print(f'Chargé depuis le CSV local : {CSV_PATH}')
else:
    print('CSV introuvable -> chargement depuis BigQuery...')
    from google.cloud import bigquery
    client = bigquery.Client(project=PROJECT)
    df = client.query(f'SELECT * FROM `{VIEW}`').to_dataframe()
    os.makedirs('data', exist_ok=True)
    df.to_csv(CSV_PATH, index=False, encoding='utf-8-sig')
    print(f'Sauvegardé en local : {CSV_PATH}')

print(f'\nDimensions : {df.shape[0]:,} lignes x {df.shape[1]} colonnes')
df.head(3)

## 3. Vue d'ensemble

In [ ]:
# Types + valeurs manquantes
apercu = pd.DataFrame({
    'type'    : df.dtypes.astype(str),
    'n_manquant': df.isnull().sum(),
    'pct_null'  : (df.isnull().mean() * 100).round(2),
    'n_unique'  : df.nunique()
})
print('=== APERÇU DES COLONNES ===')
apercu

In [ ]:
# Colonnes avec valeurs manquantes uniquement
manq = apercu[apercu['n_manquant'] > 0].sort_values('pct_null', ascending=False)
if len(manq):
    print('Colonnes avec valeurs manquantes :')
    display(manq)
else:
    print('Aucune valeur manquante (les COALESCE de la vue SQL ont fait leur travail).')

In [ ]:
# Statistiques descriptives des variables numériques
df.describe().T.round(2)

## 4. Variable cible : `y_annulation`

In [ ]:
counts = df['y_annulation'].value_counts().sort_index()
pct    = df['y_annulation'].value_counts(normalize=True).sort_index() * 100

print('=== DISTRIBUTION DE LA CIBLE ===')
print(f"  Y=0 (maintenu)  : {counts[0]:,} ({pct[0]:.2f}%)")
print(f"  Y=1 (annulé)    : {counts[1]:,} ({pct[1]:.2f}%)")
print(f"\n  Ratio de déséquilibre : 1 annulation pour {counts[0]/counts[1]:.1f} maintiens")

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['Maintenu (0)', 'Annulé (1)'], counts.values, color=[C_MAINT, C_ANNUL])
ax.set_title('Distribution de la variable cible')
ax.set_ylabel('Nombre de dossiers')
for b, v, p in zip(bars, counts.values, pct.values):
    ax.text(b.get_x()+b.get_width()/2, v, f'{v:,}\n({p:.1f}%)', ha='center', va='bottom', fontweight='bold')
ax.margins(y=0.15)
plt.tight_layout(); plt.savefig('figures/01_cible.png', dpi=150); plt.show()

## 5. La condition d'annulation (Flex) — variable structurelle clé
On attend un gradient : plus la couverture est forte, plus on annule.

In [ ]:
def taux_par(col, min_n=30, top=None):
    """Taux d'annulation + volume par modalité d'une variable catégorielle."""
    g = (df.groupby(col)['y_annulation']
           .agg(nb='count', taux_annul='mean')
           .assign(taux_annul=lambda x: (x['taux_annul']*100).round(2))
           .query('nb >= @min_n')
           .sort_values('taux_annul', ascending=False))
    return g.head(top) if top else g

cond = taux_par('cond_annulation')
print('=== TAUX D\'ANNULATION PAR CONDITION FLEX ===')
print(cond)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(cond.index, cond['taux_annul'], color=C_PURP)
ax.invert_yaxis()
ax.set_xlabel('Taux d\'annulation (%)')
ax.set_title('Taux d\'annulation par condition d\'annulation (Flex)')
for b, v in zip(bars, cond['taux_annul']):
    ax.text(v+0.2, b.get_y()+b.get_height()/2, f'{v:.1f}%', va='center', fontsize=9)
plt.tight_layout(); plt.savefig('figures/02_cond_annulation.png', dpi=150); plt.show()

In [ ]:
# Synthèse couvert vs non couvert
if 'est_assure_annulation' in df.columns:
    g = (df.groupby('est_assure_annulation')['y_annulation']
           .agg(nb='count', taux='mean'))
    g['taux'] = (g['taux']*100).round(2)
    g.index = ['Non couvert (0)', 'Couvert (1)']
    print('=== COUVERT vs NON COUVERT ===')
    print(g)

## 6. Le canal de vente

In [ ]:
canal = taux_par('canal', top=12)
print(canal)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(canal.index, canal['taux_annul'], color=C_BLUE)
ax.invert_yaxis()
ax.set_xlabel('Taux d\'annulation (%)')
ax.set_title('Taux d\'annulation par canal de vente')
for b, v, n in zip(bars, canal['taux_annul'], canal['nb']):
    ax.text(v+0.2, b.get_y()+b.get_height()/2, f'{v:.1f}% (n={n:,})', va='center', fontsize=8)
plt.tight_layout(); plt.savefig('figures/03_canal.png', dpi=150); plt.show()

## 7. Saisonnalité & destination

In [ ]:
# Thème de station
if 'theme_station' in df.columns:
    theme = taux_par('theme_station', top=10)
    print('=== PAR THÈME DE STATION ===')
    print(theme)
    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.barh(theme.index, theme['taux_annul'], color='#16a085')
    ax.invert_yaxis(); ax.set_xlabel('Taux d\'annulation (%)')
    ax.set_title('Taux d\'annulation par thème de station')
    for b, v in zip(bars, theme['taux_annul']):
        ax.text(v+0.1, b.get_y()+b.get_height()/2, f'{v:.1f}%', va='center', fontsize=9)
    plt.tight_layout(); plt.savefig('figures/04_theme.png', dpi=150); plt.show()

In [ ]:
# Période de départ
if 'periode_depart' in df.columns:
    per = taux_par('periode_depart', top=15)
    print('=== PAR PÉRIODE DE DÉPART ===')
    print(per)

In [ ]:
# Mois de réservation (saisonnalité de la prise de décision)
mois = (df.groupby('mois_resa')['y_annulation']
          .agg(nb='count', taux='mean').assign(taux=lambda x:(x['taux']*100).round(2)))
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(mois.index, mois['taux'], marker='o', color=C_ANNUL, lw=2)
ax.set_xticks(range(1,13))
ax.set_xlabel('Mois de réservation'); ax.set_ylabel('Taux d\'annulation (%)')
ax.set_title('Taux d\'annulation selon le mois de réservation')
plt.tight_layout(); plt.savefig('figures/05_mois.png', dpi=150); plt.show()

## 8. Variables numériques de contexte

In [ ]:
num_context = ['anticipation_jours', 'duree_sejour', 'dossier_nb_pax_total',
               'dossier_nb_pax_adultes', 'nb_mineur', 'nb_bebe',
               'nb_dossiers_anterieurs']
num_context = [c for c in num_context if c in df.columns]

fig, axes = plt.subplots(2, 4, figsize=(18, 8)); axes = axes.flatten()
for i, col in enumerate(num_context):
    data = df[col].clip(upper=df[col].quantile(0.99))  # coupe les outliers extrêmes pour l'affichage
    axes[i].hist(data, bins=40, color=C_BLUE, edgecolor='white')
    axes[i].set_title(col, fontsize=10)
for j in range(len(num_context), len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Distribution des variables de contexte (99e percentile)', y=1.01, fontsize=13)
plt.tight_layout(); plt.savefig('figures/06_num_context.png', dpi=150); plt.show()

In [ ]:
# Anticipation : annulés vs maintenus (un signal souvent fort)
fig, ax = plt.subplots(figsize=(9, 4))
for y, c, lab in [(0, C_MAINT, 'Maintenu'), (1, C_ANNUL, 'Annulé')]:
    d = df.loc[df['y_annulation']==y, 'anticipation_jours'].clip(upper=365)
    ax.hist(d, bins=50, alpha=0.55, color=c, label=lab, density=True)
ax.set_xlabel('Anticipation (jours, plafonné à 365)'); ax.set_ylabel('Densité')
ax.set_title('Anticipation de réservation : annulés vs maintenus')
ax.legend()
plt.tight_layout(); plt.savefig('figures/07_anticipation.png', dpi=150); plt.show()

## 9. Couverture email (CRM) — la contrainte Batch
Rappel : Batch ne couvre que ~7 mois, donc une minorité de dossiers a un historique email.

In [ ]:
if 'est_dans_crm' in df.columns:
    n_crm = int(df['est_dans_crm'].sum())
    pct_crm = df['est_dans_crm'].mean()*100
    print(f"Dossiers AVEC historique email : {n_crm:,} ({pct_crm:.1f}%)")
    print(f"Dossiers SANS historique email : {len(df)-n_crm:,} ({100-pct_crm:.1f}%)")

# Distribution du segment d'engagement
seg = df['segment_email'].value_counts()
print('\n=== RÉPARTITION DES SEGMENTS EMAIL ===')
print(seg)

In [ ]:
# Taux d'annulation par segment email (toute la population)
seg_taux = taux_par('segment_email')
print(seg_taux)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(seg_taux.index, seg_taux['taux_annul'], color=C_PURP)
ax.invert_yaxis(); ax.set_xlabel('Taux d\'annulation (%)')
ax.set_title('Taux d\'annulation par segment email (population globale)')
for b, v in zip(bars, seg_taux['taux_annul']):
    ax.text(v+0.1, b.get_y()+b.get_height()/2, f'{v:.1f}%', va='center', fontsize=9)
plt.tight_layout(); plt.savefig('figures/08_segment_global.png', dpi=150); plt.show()

## 10. ★ ANALYSE CENTRALE : l'email chez les clients NON couverts
C'est le cœur du mémoire : chez ceux qui perdent leur argent en annulant (non assurés), l'engagement email discrimine-t-il l'annulation ?

In [ ]:
if 'est_assure_annulation' in df.columns:
    non_couverts = df[df['est_assure_annulation'] == 0]
    print(f'Population non couverte : {len(non_couverts):,} dossiers '
          f'(taux annul. global {non_couverts["y_annulation"].mean()*100:.2f}%)\n')

    g = (non_couverts.groupby('segment_email')['y_annulation']
                     .agg(nb='count', taux='mean')
                     .assign(taux=lambda x:(x['taux']*100).round(2))
                     .sort_values('taux', ascending=False))
    print('=== TAUX D\'ANNULATION PAR SEGMENT EMAIL (NON COUVERTS) ===')
    print(g)

    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.barh(g.index, g['taux'], color=C_ANNUL)
    ax.invert_yaxis(); ax.set_xlabel('Taux d\'annulation (%)')
    ax.set_title('★ Annulation par engagement email — clients NON couverts')
    for b, v, n in zip(bars, g['taux'], g['nb']):
        ax.text(v+0.1, b.get_y()+b.get_height()/2, f'{v:.1f}% (n={n:,})', va='center', fontsize=8)
    plt.tight_layout(); plt.savefig('figures/09_email_non_couverts.png', dpi=150); plt.show()

## 11. Matrice de corrélation (variables numériques)

In [ ]:
corr_cols = ['y_annulation', 'anticipation_jours', 'duree_sejour',
             'dossier_nb_pax_total', 'nb_dossiers_anterieurs',
             'nb_clics_90j', 'nb_ouvertures_90j', 'nb_campagnes_recues',
             'recence_email_jours', 'taux_clic_sur_ouverture', 'a_interagi_email',
             'est_assure_annulation', 'est_client_vip', 'a_promo']
corr_cols = [c for c in corr_cols if c in df.columns]

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(df[corr_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=False, cbar_kws={'shrink':0.8}, ax=ax)
ax.set_title('Matrice de corrélation')
plt.tight_layout(); plt.savefig('figures/10_correlation.png', dpi=150); plt.show()

In [ ]:
# Corrélations avec la cible, triées
corr_cible = df[corr_cols].corr()['y_annulation'].drop('y_annulation').sort_values(key=abs, ascending=False)
print('=== CORRÉLATION (Pearson) AVEC LA CIBLE ===')
print(corr_cible.round(3))

## 12. Synthèse EDA

In [ ]:
print('='*55)
print('  SYNTHÈSE EDA — DATASET ANNULATION')
print('='*55)
print(f'  Dossiers          : {len(df):,}')
print(f'  Variables         : {df.shape[1]}')
print(f'  Taux annulation   : {df["y_annulation"].mean()*100:.2f}%')
if 'est_dans_crm' in df.columns:
    print(f'  Couverture email  : {df["est_dans_crm"].mean()*100:.1f}%')
print()
print('  À retenir pour la modélisation :')
print('   - Déséquilibre modéré -> class_weight / SMOTE')
print('   - cond_annulation & canal = signaux structurels forts')
print('   - Email : signal réel mais sur une minorité (sous-ensemble CRM)')
print('   - Stratégie 2 temps : modèle global + focus est_dans_crm=1')
print('  Graphiques sauvegardés dans ./figures/')
print('='*55)